# Qwen3.5-9B

![alt](https://sebastianraschka.com/llm-architecture-gallery/images/architectures/qwen3-5-397b.webp)
---
> Qwen3.5 397B model architecture. The 9B variant is similar in its layout.

This notebook serves as a simple working example of [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B), implemented by hand.

Qwen3.5 is Alibaba's open hybrid family. Most layers are **Gated DeltaNet** (linear attention / delta-rule recurrence), interleaved 3:1 with **gated GQA**.
The 9B dense model is multimodal and ships a native **multi-token prediction (MTP)** head for speculative decoding. Context is 262K native.

It was chosen because:

- At Q8 the weights comfortably fit an RTX 4080 (16 GB) when quantised.
- Gated DeltaNet is used in a number of production hybrids as is an interesting challenge to implement.
- The built in MTP head allows for speculative decode to be implemented and experimented with.

Other models considered:

- [Gemma 4 12B](https://huggingface.co/google/gemma-4-12B-it) is an obvious transformer alternative (hybrid sliding-window GQA and MTP) but the interesting kernel there is a windowed KV cache. This is less of a reason to write an engine.
- [LFM 2.5 8B-A1B](https://huggingface.co/LiquidAI/LFM2.5-8B-A1B) is also an interesting model but is quite "exotic". This architecture is not found in many production models.

In [1]:
import os
import warnings
import logging
from tqdm import TqdmWarning

# Suppress warnings from Huggingface
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore", message=".*unauthenticated requests to the HF Hub.*")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

MODEL = "Qwen/Qwen3.5-9B"

## Tokenising Text

In [2]:
from tokenizers import Tokenizer

tok = Tokenizer.from_pretrained(MODEL)

### Tokenisation Pipeline: Normalisation

Text first must be "normalised" to make it less random and clean. This includes stripping out whitespace, stripping accented characters or lowercasing all text.

In [3]:
from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents

For example:

In [4]:
norm = normalizers.Sequence([NFD(), StripAccents()])
pre_norm_text = "Héllò hôw are ü?"
norm_text = norm.normalize_str(pre_norm_text)

print(f"Pre-normed text: {pre_norm_text}")
print(f"Normalised text: {norm_text}")

Pre-normed text: Héllò hôw are ü?
Normalised text: Hello how are u?


This isn't what Qwen does, it just does `NFD`. This is just to illustrate the point.

### Tokenisation Pipeline: Pre-Tokenisation

Pre-tokenisation is the act of splitting text into smaller objects.

For a simple example, text can be split on spaces and punctuation.

In [5]:
from tokenizers.pre_tokenizers import Whitespace

pre_tokenizer = Whitespace()
pre_token = pre_tokenizer.pre_tokenize_str("Hello! How are you? I'm fine, thank you.")

print("Pre-tokenised text (token | start idx | end idx):")
pre_token

Pre-tokenised text (token | start idx | end idx):


[('Hello', (0, 5)),
 ('!', (5, 6)),
 ('How', (7, 10)),
 ('are', (11, 14)),
 ('you', (15, 18)),
 ('?', (18, 19)),
 ('I', (20, 21)),
 ("'", (21, 22)),
 ('m', (22, 23)),
 ('fine', (24, 28)),
 (',', (28, 29)),
 ('thank', (30, 35)),
 ('you', (36, 39)),
 ('.', (39, 40))]

This isn't what Qwen does, it just does `NFD`. This is just to illustrate the point.

### Tokenisation Pipeline: Model

Once the input texts are normalised and pre-tokenised, the tokeniser is applies the model rules on the pre-tokens. The role of the model is the split the "words" into tokens using the rules it learnt.
It will also be responsible for mapping the tokens to their corresponding IDs in the vocab of the model.

In [6]:
enc = tok.encode("Hello! How are you? I'm fine, thank you.")

In [7]:
print("Pre-tokenised:")
enc.tokens

Pre-tokenised:


['Hello',
 '!',
 'ĠHow',
 'Ġare',
 'Ġyou',
 '?',
 'ĠI',
 "'m",
 'Ġfine',
 ',',
 'Ġthank',
 'Ġyou',
 '.']

Note here how spaces have been converted to `Ġ`.

In [8]:
print("Token IDs:")
enc.ids

Token IDs:


[9419, 0, 2500, 513, 488, 30, 353, 2688, 6699, 11, 9414, 488, 13]

These are the BPE leant IDs of the above pre-tokenised text.

### Tokenisation Pipeline: Post-Processing

This is the last step. This is to add on potential special tokens. With Qwen (and most new LLMs), this is not needed. This is because a chat template is what inserts the special tokens.

In [9]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello, world!"},
]
prompt = tok.apply_chat_template(
    messages,
    tokenize=False,              # string, not IDs
    add_generation_prompt=True,  # end with assistant turn so the model replies
    enable_thinking=True,        # Qwen3.5 default: open a <think> block
)

In [10]:
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello, world!<|im_end|>
<|im_start|>assistant
<think>



Note how `<|im_start|>` and `<think>` tokens have been automatically placed in.

In [11]:
model_vocab = tok.get_vocab()
speicals = sorted(
    ((i, t) for t, i in model_vocab.items() if t.startswith("<") and t.endswith(">")),
    key=lambda x: x[0]
)

print("ID\t|\tSpecial Token")
for i, t in speicals:
    print(f"{i:,}\t{t}")

ID	|	Special Token
18,947	<?>
20,475	<>
69,262	<()>
71,574	<?>>
248,044	<|endoftext|>
248,045	<|im_start|>
248,046	<|im_end|>
248,047	<|object_ref_start|>
248,048	<|object_ref_end|>
248,049	<|box_start|>
248,050	<|box_end|>
248,051	<|quad_start|>
248,052	<|quad_end|>
248,053	<|vision_start|>
248,054	<|vision_end|>
248,055	<|vision_pad|>
248,056	<|image_pad|>
248,057	<|video_pad|>
248,058	<tool_call>
248,059	</tool_call>
248,060	<|fim_prefix|>
248,061	<|fim_middle|>
248,062	<|fim_suffix|>
248,063	<|fim_pad|>
248,064	<|repo_name|>
248,065	<|file_sep|>
248,066	<tool_response>
248,067	</tool_response>
248,068	<think>
248,069	</think>
248,070	<|audio_start|>
248,071	<|audio_end|>
248,072	<tts_pad>
248,073	<tts_text_bos>
248,074	<tts_text_eod>
248,075	<tts_text_bos_single>
248,076	<|audio_pad|>


### Conclusion

The tokenisation pipeline goes like this:
1. Normalisation
2. Pre-tokenisation
3. Model rules
4. Post-processing

To read more, read [Huggingface's page on the tokenisation pipeline](https://huggingface.co/docs/tokenizers/pipeline).